In [1]:
!pip install lightgbm catboost xgboost -q

import os, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
!ls /kaggle/input/datasets/tymofiivoitekh/assignment-3/

sample_submission.csv  test  train


In [3]:
TRAIN_DIR = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/train/train')
TEST_DIR  = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/test/test')
SAMPLE    = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/sample_submission.csv')

# verify
print(len(list(TRAIN_DIR.rglob('*.csv'))), 'train files')
print(len(list(TEST_DIR.rglob('*.csv'))),  'test files')

11020 train files
6849 test files


In [4]:
def extract_features(df):
    feats = {}
    axes = ['mean_x','mean_y','mean_z','std_x','std_y','std_z']
    
    for col in axes:
        s = df[col].values
        feats[f'{col}_mean']    = np.mean(s)
        feats[f'{col}_std']     = np.std(s)
        feats[f'{col}_min']     = np.min(s)
        feats[f'{col}_max']     = np.max(s)
        feats[f'{col}_range']   = np.ptp(s)
        feats[f'{col}_median']  = np.median(s)
        feats[f'{col}_q25']     = np.percentile(s, 25)
        feats[f'{col}_q75']     = np.percentile(s, 75)
        feats[f'{col}_iqr']     = np.percentile(s, 75) - np.percentile(s, 25)
        feats[f'{col}_skew']    = stats.skew(s)
        feats[f'{col}_kurt']    = stats.kurtosis(s)
        feats[f'{col}_energy']  = np.mean(s**2)
        feats[f'{col}_rms']     = np.sqrt(np.mean(s**2))
        feats[f'{col}_cv']      = np.std(s) / (np.mean(s) + 1e-9)
        feats[f'{col}_mad']     = np.mean(np.abs(s - np.mean(s)))
        feats[f'{col}_sum_abs'] = np.sum(np.abs(s))
        feats[f'{col}_zero_cr'] = np.mean(np.diff(np.sign(s)) != 0)
        feats[f'{col}_entropy'] = stats.entropy(np.abs(s) + 1e-9)
        # autocorrelation lag-1
        if len(s) > 1:
            feats[f'{col}_autocorr1'] = np.corrcoef(s[:-1], s[1:])[0,1] if np.std(s) > 0 else 0
        # trend (slope of linear fit)
        feats[f'{col}_slope'] = np.polyfit(np.arange(len(s)), s, 1)[0]
        # first & last 30-second averages (30 rows = ~30s)
        feats[f'{col}_first30_mean'] = np.mean(s[:30])
        feats[f'{col}_last30_mean']  = np.mean(s[-30:])
        feats[f'{col}_mid_mean']     = np.mean(s[135:165])

    # cross-axis features
    mx, my, mz = df['mean_x'].values, df['mean_y'].values, df['mean_z'].values
    magnitude = np.sqrt(mx**2 + my**2 + mz**2)
    feats['mag_mean']   = np.mean(magnitude)
    feats['mag_std']    = np.std(magnitude)
    feats['mag_max']    = np.max(magnitude)
    feats['mag_min']    = np.min(magnitude)
    feats['mag_range']  = np.ptp(magnitude)
    feats['mag_energy'] = np.mean(magnitude**2)
    feats['mag_skew']   = stats.skew(magnitude)
    feats['mag_kurt']   = stats.kurtosis(magnitude)
    feats['mag_entropy']= stats.entropy(magnitude + 1e-9)

    # jerk (derivative of mean acceleration)
    for col, arr in [('x', mx), ('y', my), ('z', mz)]:
        jerk = np.diff(arr)
        feats[f'jerk_{col}_mean']  = np.mean(np.abs(jerk))
        feats[f'jerk_{col}_std']   = np.std(jerk)
        feats[f'jerk_{col}_max']   = np.max(np.abs(jerk))
        feats[f'jerk_{col}_energy']= np.mean(jerk**2)

    mag_jerk = np.diff(magnitude)
    feats['mag_jerk_mean']  = np.mean(np.abs(mag_jerk))
    feats['mag_jerk_std']   = np.std(mag_jerk)
    feats['mag_jerk_energy']= np.mean(mag_jerk**2)

    # correlations between axes
    feats['corr_xy'] = np.corrcoef(mx, my)[0,1] if (np.std(mx)>0 and np.std(my)>0) else 0
    feats['corr_xz'] = np.corrcoef(mx, mz)[0,1] if (np.std(mx)>0 and np.std(mz)>0) else 0
    feats['corr_yz'] = np.corrcoef(my, mz)[0,1] if (np.std(my)>0 and np.std(mz)>0) else 0

    # frequency domain (FFT)
    for col, arr in [('x', mx), ('y', my), ('z', mz), ('mag', magnitude)]:
        fft_vals = np.abs(np.fft.rfft(arr))
        feats[f'fft_{col}_max']      = np.max(fft_vals)
        feats[f'fft_{col}_mean']     = np.mean(fft_vals)
        feats[f'fft_{col}_std']      = np.std(fft_vals)
        feats[f'fft_{col}_dominant'] = np.argmax(fft_vals[1:]) + 1  # dominant freq (skip DC)
        feats[f'fft_{col}_energy']   = np.sum(fft_vals**2)
        # spectral entropy
        psd = fft_vals**2
        psd_norm = psd / (psd.sum() + 1e-9)
        feats[f'fft_{col}_spec_entropy'] = -np.sum(psd_norm * np.log(psd_norm + 1e-9))

    # segment statistics (split 300s into 6 windows of 50s each)
    n_segs = 6
    seg_len = len(df) // n_segs
    for seg_i in range(n_segs):
        seg = df.iloc[seg_i*seg_len:(seg_i+1)*seg_len]
        seg_mag = np.sqrt(seg['mean_x']**2 + seg['mean_y']**2 + seg['mean_z']**2)
        feats[f'seg{seg_i}_mag_mean'] = seg_mag.mean()
        feats[f'seg{seg_i}_mag_std']  = seg_mag.std()
        feats[f'seg{seg_i}_mean_x']   = seg['mean_x'].mean()
        feats[f'seg{seg_i}_mean_y']   = seg['mean_y'].mean()
        feats[f'seg{seg_i}_mean_z']   = seg['mean_z'].mean()

    return feats

In [5]:
def load_dataset(folder):
    records = []
    for fpath in sorted(folder.rglob('*.csv')):
        df = pd.read_csv(fpath)
        fid = int(df['file_id'].iloc[0])
        label = int(df['label'].iloc[0]) if 'label' in df.columns else -1
        feats = extract_features(df)
        feats['file_id'] = fid
        feats['label']   = label
        records.append(feats)
    return pd.DataFrame(records)

print('Loading train...')
train_df = load_dataset(TRAIN_DIR)
print(f'Train shape: {train_df.shape}')

print('Loading test...')
test_df = load_dataset(TEST_DIR)
print(f'Test shape: {test_df.shape}')

Loading train...


KeyboardInterrupt: 

In [ ]:
feature_cols = [c for c in train_df.columns if c not in ['file_id', 'label']]

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df['label'].values.astype(int)

X_test  = test_df[feature_cols].values.astype(np.float32)
test_ids = test_df['file_id'].values.astype(int)

# replace NaN / inf
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

print('X_train:', X_train.shape, '| classes:', np.unique(y_train))
print('X_test: ', X_test.shape)

In [6]:
import os, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
warnings.filterwarnings('ignore')
np.random.seed(42)

TRAIN_DIR = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/train/train')
TEST_DIR  = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/test/test')
SAMPLE    = Path('/kaggle/input/datasets/tymofiivoitekh/assignment-3/sample_submission.csv')

train_files = sorted(TRAIN_DIR.rglob('*.csv'))
test_files  = sorted(TEST_DIR.rglob('*.csv'))

print(f"Train files : {len(train_files)}")
print(f"Test files  : {len(test_files)}")
print(f"\nSample train path: {train_files[0]}")
print(f"Sample test  path: {test_files[0]}")

# Peek at one file
df_sample = pd.read_csv(train_files[0])
print(f"\nColumns: {list(df_sample.columns)}")
print(f"Shape  : {df_sample.shape}")
print(df_sample.head(5))

Train files : 11020
Test files  : 6849

Sample train path: /kaggle/input/datasets/tymofiivoitekh/assignment-3/train/train/User_001/00001.csv
Sample test  path: /kaggle/input/datasets/tymofiivoitekh/assignment-3/test/test/User_061/11021.csv

Columns: ['index', 'mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z', 'label', 'file_id']
Shape  : (300, 9)
   index    mean_x    mean_y    mean_z         std_x     std_y     std_z  \
0      0 -0.467629 -0.537854  0.657240  3.733827e-03  0.007097  0.004734   
1      1 -0.466690 -0.547035  0.654931  1.115816e-16  0.005082  0.006511   
2      2 -0.467316 -0.535987  0.658472  3.080919e-03  0.005875  0.002188   
3      3 -0.468880 -0.533341  0.658472  5.455416e-03  0.000000  0.000000   
4      4 -0.470288 -0.533341  0.658472  6.616433e-03  0.000000  0.000000   

   label  file_id  
0      0        1  
1      0        1  
2      0        1  
3      0        1  
4      0        1  


In [ ]:
print("Scanning ALL train files for length/NaN anomalies...")

lengths, labels, file_ids, short_files = [], [], [], []

for fpath in train_files:
    df = pd.read_csv(fpath)
    n  = len(df)
    lengths.append(n)
    if n != 300:
        short_files.append((str(fpath), n))
    label = int(df['label'].iloc[0]) if 'label' in df.columns else -1
    fid   = int(df['file_id'].iloc[0])
    labels.append(label)
    file_ids.append(fid)

lengths = np.array(lengths)
print(f"\n--- Row count stats (expected 300) ---")
print(f"  Min   : {lengths.min()}")
print(f"  Max   : {lengths.max()}")
print(f"  Mean  : {lengths.mean():.2f}")
print(f"  Files != 300 rows: {len(short_files)}")
if short_files:
    for fp, l in short_files[:10]:
        print(f"    {fp}: {l} rows")

# NaN check on sample of 200 files
print("\n--- NaN check (200-file sample) ---")
total_nans = 0
for fpath in train_files[:200]:
    df = pd.read_csv(fpath)
    total_nans += df.isna().sum().sum()
print(f"  Total NaNs in 200-file sample: {total_nans}")

# Test file lengths
test_lengths = []
for fpath in test_files[:300]:
    df = pd.read_csv(fpath)
    test_lengths.append(len(df))
test_lengths = np.array(test_lengths)
print(f"\n--- Test file row count stats (first 300 files) ---")
print(f"  Min: {test_lengths.min()}, Max: {test_lengths.max()}, Mean: {test_lengths.mean():.2f}")

Scanning ALL train files for length/NaN anomalies...

--- Row count stats (expected 300) ---
  Min   : 300
  Max   : 300
  Mean  : 300.00
  Files != 300 rows: 0

--- NaN check (200-file sample) ---
  Total NaNs in 200-file sample: 0

--- Test file row count stats (first 300 files) ---
  Min: 300, Max: 300, Mean: 300.00


In [ ]:
label_series = pd.Series(labels)
counts = label_series.value_counts().sort_index()

print("=== Training Label Distribution ===")
for lbl, cnt in counts.items():
    bar = '█' * int(cnt / counts.max() * 40)
    print(f"  Label {lbl}: {cnt:5d} ({cnt/len(labels)*100:5.1f}%)  {bar}")

print(f"\nTotal files      : {len(labels)}")
print(f"Imbalance ratio  : {counts.max() / counts.min():.2f}x  (max/min)")
print(f"Minority classes : {list(counts[counts < counts.mean()].index)}")
print(f"Majority classes : {list(counts[counts >= counts.mean()].index)}")

# Files per user
user_dirs = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
print(f"\nTotal users: {len(user_dirs)}")
files_per_user = [len(list(u.glob('*.csv'))) for u in user_dirs]
print(f"Files per user: min={min(files_per_user)}, max={max(files_per_user)}, mean={np.mean(files_per_user):.1f}")

=== Training Label Distribution ===
  Label 0:  4643 ( 42.1%)  ███████████████████████████████████████
  Label 1:  4695 ( 42.6%)  ████████████████████████████████████████
  Label 2:   358 (  3.2%)  ███
  Label 3:   656 (  6.0%)  █████
  Label 4:   142 (  1.3%)  █
  Label 5:   526 (  4.8%)  ████

Total files      : 11020
Imbalance ratio  : 33.06x  (max/min)
Minority classes : [2, 3, 4, 5]
Majority classes : [0, 1]

Total users: 60
Files per user: min=108, max=256, mean=183.7


In [ ]:
user_label_map = {}  # user -> list of labels

for fpath in train_files:
    user = fpath.parent.name
    df   = pd.read_csv(fpath)
    lbl  = int(df['label'].iloc[0]) if 'label' in df.columns else -1
    user_label_map.setdefault(user, []).append(lbl)

distinct_per_user = {u: len(set(lbls)) for u, lbls in user_label_map.items()}

print("=== Activities per user ===")
print(f"  Min distinct activities : {min(distinct_per_user.values())}")
print(f"  Max distinct activities : {max(distinct_per_user.values())}")
print(f"  Mean distinct activities: {np.mean(list(distinct_per_user.values())):.2f}")

# How many users have each activity?
print("\n=== Users covering each class ===")
from collections import Counter
class_user_count = {c: 0 for c in range(6)}
for u, lbls in user_label_map.items():
    for c in set(lbls):
        class_user_count[c] += 1
for c, cnt in sorted(class_user_count.items()):
    print(f"  Class {c}: covered by {cnt} users")

# If a model is split by file (not user), same-user files may leak into val
# Check: do any users appear in test folder?
test_users = set(f.parent.name for f in test_files)
train_users = set(user_label_map.keys())
overlap = test_users & train_users
print(f"\nUser overlap between train and test dirs: {len(overlap)} users")
print(f"  Train-only users: {len(train_users - test_users)}")
print(f"  Test-only users : {len(test_users - train_users)}")
print(f"  Shared users    : {len(overlap)}")
if overlap:
    print(f"  Example shared  : {list(overlap)[:5]}")

=== Activities per user ===
  Min distinct activities : 2
  Max distinct activities : 6
  Mean distinct activities: 4.75

=== Users covering each class ===
  Class 0: covered by 60 users
  Class 1: covered by 60 users
  Class 2: covered by 52 users
  Class 3: covered by 59 users
  Class 4: covered by 19 users
  Class 5: covered by 35 users

User overlap between train and test dirs: 0 users
  Train-only users: 60
  Test-only users : 40
  Shared users    : 0


In [ ]:
from collections import defaultdict

class_files = defaultdict(list)
for fpath, lbl in zip(train_files, labels):
    class_files[lbl].append(fpath)

axes = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']
SAMPLE_N = 30  # files per class to sample

print("=== Mean Signal Statistics Per Class ===")
print(f"(averaged over {SAMPLE_N} files per class)\n")

class_stats = {}  # store for later use

for cls in sorted(class_files.keys()):
    fpaths = class_files[cls]
    sample = fpaths[:SAMPLE_N]
    
    stat_rows = []
    for fp in sample:
        df  = pd.read_csv(fp)
        mag = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2)
        jerk = np.abs(np.diff(mag.values))
        row = {
            'mag_mean'  : mag.mean(),
            'mag_std'   : mag.std(),
            'mag_range' : mag.max() - mag.min(),
            'jerk_mean' : jerk.mean(),
            'jerk_max'  : jerk.max(),
            'std_x_mean': df['std_x'].mean(),   # within-second sensor noise
            'std_y_mean': df['std_y'].mean(),
            'std_z_mean': df['std_z'].mean(),
        }
        for ax in ['mean_x', 'mean_y', 'mean_z']:
            row[f'{ax}_mean'] = df[ax].mean()
            row[f'{ax}_std']  = df[ax].std()
        stat_rows.append(row)
    
    agg = pd.DataFrame(stat_rows).mean()
    class_stats[cls] = agg
    
    print(f"--- Class {cls}  ({len(fpaths)} files total) ---")
    print(f"  mag_mean   : {agg['mag_mean']:.4f}   (gravity ≈ 1.0g means sensor is near-static)")
    print(f"  mag_std    : {agg['mag_std']:.4f}   (higher = more movement)")
    print(f"  mag_range  : {agg['mag_range']:.4f}")
    print(f"  jerk_mean  : {agg['jerk_mean']:.5f}  (rate of accel change)")
    print(f"  jerk_max   : {agg['jerk_max']:.5f}")
    print(f"  std_x_mean : {agg['std_x_mean']:.4f}  (within-second noise X)")
    print(f"  mean_x_mean: {agg['mean_x_mean']:+.4f}")
    print(f"  mean_y_mean: {agg['mean_y_mean']:+.4f}")
    print(f"  mean_z_mean: {agg['mean_z_mean']:+.4f}")
    print()

=== Mean Signal Statistics Per Class ===
(averaged over 30 files per class)

--- Class 0  (4643 files total) ---
  mag_mean   : 0.9878   (gravity ≈ 1.0g means sensor is near-static)
  mag_std    : 0.0024   (higher = more movement)
  mag_range  : 0.0187
  jerk_mean  : 0.00072  (rate of accel change)
  jerk_max   : 0.01562
  std_x_mean : 0.0041  (within-second noise X)
  mean_x_mean: -0.6603
  mean_y_mean: -0.3987
  mean_z_mean: +0.3502

--- Class 1  (4695 files total) ---
  mag_mean   : 0.9603   (gravity ≈ 1.0g means sensor is near-static)
  mag_std    : 0.0369   (higher = more movement)
  mag_range  : 0.3268
  jerk_mean  : 0.01524  (rate of accel change)
  jerk_max   : 0.25570
  std_x_mean : 0.0508  (within-second noise X)
  mean_x_mean: -0.5760
  mean_y_mean: +0.2300
  mean_z_mean: +0.2983

--- Class 2  (358 files total) ---
  mag_mean   : 0.9859   (gravity ≈ 1.0g means sensor is near-static)
  mag_std    : 0.0377   (higher = more movement)
  mag_range  : 0.4382
  jerk_mean  : 0.02094

In [ ]:
def fft_band_power(arr, fs=1.0):
    arr = arr - np.mean(arr)  # remove DC
    fft_vals = np.abs(np.fft.rfft(arr))
    freqs    = np.fft.rfftfreq(len(arr), d=1.0/fs)
    psd      = fft_vals**2
    total    = psd.sum() + 1e-9

    # Frequency bands (Hz = cycles/second in the 5-min sequence)
    bands = {
        'DC_removed': (freqs > 0,   freqs < 0.005),   # near-DC
        'very_low'  : (freqs >= 0.005, freqs < 0.02),
        'low'       : (freqs >= 0.02,  freqs < 0.05),
        'mid'       : (freqs >= 0.05,  freqs < 0.1),
        'high'      : (freqs >= 0.1,   freqs <= 0.5),
    }
    result = {}
    for bname, (lo, hi) in bands.items():
        mask = lo & hi
        result[f'{bname}_ratio'] = psd[mask].sum() / total
    result['dominant_freq'] = freqs[np.argmax(fft_vals[1:]) + 1]
    
    # Spectral entropy (how spread out the energy is)
    psd_norm = psd / total
    result['spectral_entropy'] = -np.sum(psd_norm * np.log(psd_norm + 1e-9))
    return result

print("=== FFT Band Power Ratios Per Class ===")
print(f"{'Cls':<5} {'vlow_r':>8} {'low_r':>8} {'mid_r':>8} {'high_r':>8} {'dom_hz':>9} {'spec_ent':>10}")

for cls in sorted(class_files.keys()):
    fpaths = class_files[cls][:30]
    results = []
    for fp in fpaths:
        df  = pd.read_csv(fp)
        mag = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2).values
        r   = fft_band_power(mag)
        results.append(r)
    agg = pd.DataFrame(results).mean()
    print(f"  {cls}    "
          f"{agg['very_low_ratio']:>8.4f} "
          f"{agg['low_ratio']:>8.4f} "
          f"{agg['mid_ratio']:>8.4f} "
          f"{agg['high_ratio']:>8.4f} "
          f"{agg['dominant_freq']:>9.5f} "
          f"{agg['spectral_entropy']:>10.4f}")

=== FFT Band Power Ratios Per Class ===
Cls     vlow_r    low_r    mid_r   high_r    dom_hz   spec_ent
  0      0.0793   0.0559   0.0720   0.6119   0.11378     3.6867
  1      0.1888   0.1401   0.1055   0.4358   0.01333     3.9456
  2      0.0742   0.1037   0.0915   0.6785   0.11389     4.4538
  3      0.1101   0.1098   0.1062   0.6019   0.09789     4.3149
  4      0.0440   0.0517   0.0663   0.8005   0.28011     4.4982
  5      0.1558   0.1159   0.0935   0.5696   0.04789     4.2584


In [ ]:
def gravity_dynamic_features(df, window=10):
    result = {}
    for ax in ['x', 'y', 'z']:
        raw     = df[f'mean_{ax}'].values
        gravity = pd.Series(raw).rolling(window, center=True, min_periods=1).mean().values
        dynamic = raw - gravity
        result[f'grav_{ax}_mean']    = gravity.mean()
        result[f'grav_{ax}_std']     = gravity.std()
        result[f'dynamic_{ax}_mean'] = np.abs(dynamic).mean()
        result[f'dynamic_{ax}_std']  = dynamic.std()
        result[f'dynamic_{ax}_energy'] = np.mean(dynamic**2)

    # Tilt angles from the gravity vector
    gx = pd.Series(df['mean_x'].values).rolling(window*2, center=True, min_periods=1).mean().values
    gy = pd.Series(df['mean_y'].values).rolling(window*2, center=True, min_periods=1).mean().values
    gz = pd.Series(df['mean_z'].values).rolling(window*2, center=True, min_periods=1).mean().values
    g_mag = np.sqrt(gx**2 + gy**2 + gz**2) + 1e-9

    pitch = np.arcsin(np.clip(gx / g_mag, -1, 1))
    roll  = np.arcsin(np.clip(gy / g_mag, -1, 1))

    result['pitch_mean']  = pitch.mean()
    result['pitch_std']   = pitch.std()
    result['pitch_range'] = np.ptp(pitch)
    result['roll_mean']   = roll.mean()
    result['roll_std']    = roll.std()
    result['roll_range']  = np.ptp(roll)
    return result

print("=== Gravity & Tilt Features Per Class ===\n")
keys_to_show = ['grav_z_mean', 'dynamic_x_mean', 'dynamic_y_mean', 'dynamic_z_mean',
                'pitch_mean', 'pitch_std', 'roll_mean', 'roll_std']

print(f"{'Cls':<5}", end="")
for k in keys_to_show:
    print(f"{k[:12]:>13}", end="")
print()

for cls in sorted(class_files.keys()):
    fpaths = class_files[cls][:30]
    rows = []
    for fp in fpaths:
        df = pd.read_csv(fp)
        rows.append(gravity_dynamic_features(df))
    agg = pd.DataFrame(rows).mean()
    print(f"  {cls}  ", end="")
    for k in keys_to_show:
        print(f"{agg[k]:>+13.4f}", end="")
    print()

=== Gravity & Tilt Features Per Class ===

Cls    grav_z_mean dynamic_x_me dynamic_y_me dynamic_z_me   pitch_mean    pitch_std    roll_mean     roll_std
  0        +0.3502      +0.0011      +0.0016      +0.0009      -0.8321      +0.0223      -0.4258      +0.0258
  1        +0.2985      +0.0542      +0.0684      +0.0643      -0.7615      +0.2997      +0.2936      +0.3538
  2        +0.1112      +0.0573      +0.0742      +0.0803      -0.5245      +0.2612      -0.0102      +0.3738
  3        -0.0167      +0.1052      +0.1530      +0.1389      -0.8859      +0.2642      -0.3780      +0.4106
  4        +0.6366      +0.0551      +0.0529      +0.0452      -0.2016      +0.1615      -0.1967      +0.1195
  5        +0.3730      +0.0786      +0.0631      +0.0819      -0.7460      +0.2658      +0.3779      +0.2508


In [ ]:
print("=== Signal Drift Across 5-Min Window (first vs last 60s) ===")
print("(shows whether the activity signal is stable throughout)\n")

print(f"{'Cls':<5} {'mag_first60':>12} {'mag_mid60':>12} {'mag_last60':>12} {'drift':>10} {'std_of_seg_means':>18}")

for cls in sorted(class_files.keys()):
    fpaths = class_files[cls][:30]
    first60, mid60, last60, seg_stds = [], [], [], []
    
    for fp in fpaths:
        df  = pd.read_csv(fp)
        mag = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2).values
        
        # Split into 5 equal segments of 60s
        segs = [mag[i*60:(i+1)*60].mean() for i in range(5)]
        first60.append(segs[0])
        mid60.append(segs[2])
        last60.append(segs[-1])
        seg_stds.append(np.std(segs))  # how much segments vary within one file
    
    f60   = np.mean(first60)
    m60   = np.mean(mid60)
    l60   = np.mean(last60)
    drift = abs(l60 - f60)
    
    print(f"  {cls}   {f60:>12.4f} {m60:>12.4f} {l60:>12.4f} {drift:>10.4f} {np.mean(seg_stds):>18.4f}")

print("\n→ High 'drift' or 'std_of_seg_means' = temporal ordering matters → LSTM/Transformer will help")
print("→ Low drift = activity is steady → feature-based models suffice")

=== Signal Drift Across 5-Min Window (first vs last 60s) ===
(shows whether the activity signal is stable throughout)

Cls    mag_first60    mag_mid60   mag_last60      drift   std_of_seg_means
  0         0.9875       0.9882       0.9876     0.0001             0.0013
  1         0.9552       0.9602       0.9635     0.0083             0.0143
  2         0.9854       0.9856       0.9872     0.0018             0.0067
  3         0.9975       0.9971       0.9855     0.0120             0.0192
  4         0.9782       0.9819       0.9807     0.0025             0.0054
  5         0.9692       0.9741       0.9717     0.0026             0.0135

→ High 'drift' or 'std_of_seg_means' = temporal ordering matters → LSTM/Transformer will help
→ Low drift = activity is steady → feature-based models suffice


In [ ]:
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.dummy import DummyClassifier

# Build a minimal flat feature vector: just 8 stats per file (mag-based)
print("Building minimal feature matrix (all train files)...")

mini_X, mini_y = [], []
for fpath, lbl in zip(train_files, labels):
    df  = pd.read_csv(fpath)
    mag = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2).values
    jerk = np.abs(np.diff(mag))
    row = [
        mag.mean(), mag.std(), mag.min(), mag.max(),
        jerk.mean(), jerk.max(), jerk.std(),
        df['std_x'].mean(), df['std_y'].mean(), df['std_z'].mean(),
        np.mean(mag[:60]), np.mean(mag[120:180]), np.mean(mag[240:]),
    ]
    mini_X.append(row)
    mini_y.append(lbl)

mini_X = np.array(mini_X)
mini_y = np.array(mini_y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline 1: majority class
dummy = DummyClassifier(strategy='most_frequent')
scores = []
for tr, val in skf.split(mini_X, mini_y):
    dummy.fit(mini_X[tr], mini_y[tr])
    scores.append(f1_score(mini_y[val], dummy.predict(mini_X[val]), average='macro'))
print(f"Naive majority-class F1  : {np.mean(scores):.4f}")

# Baseline 2: stratified random
dummy2 = DummyClassifier(strategy='stratified')
scores2 = []
for tr, val in skf.split(mini_X, mini_y):
    dummy2.fit(mini_X[tr], mini_y[tr])
    scores2.append(f1_score(mini_y[val], dummy2.predict(mini_X[val]), average='macro'))
print(f"Naive stratified F1      : {np.mean(scores2):.4f}")

# Baseline 3: 13-feature magnitude model (Decision Tree)
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
scores3 = []
for tr, val in skf.split(mini_X, mini_y):
    dt.fit(mini_X[tr], mini_y[tr])
    scores3.append(f1_score(mini_y[val], dt.predict(mini_X[val]), average='macro'))
print(f"Decision Tree (depth=5) F1: {np.mean(scores3):.4f}")

# Baseline 4: Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
scores4 = []
for tr, val in skf.split(mini_X, mini_y):
    rf.fit(mini_X[tr], mini_y[tr])
    scores4.append(f1_score(mini_y[val], rf.predict(mini_X[val]), average='macro'))
print(f"Random Forest (100 trees) F1: {np.mean(scores4):.4f}")

print("\n→ These baselines tell you the floor.")
print("→ Your current 0.77 is above these — now we need to find the ceiling.")

Building minimal feature matrix (all train files)...
Naive majority-class F1  : 0.0996
Naive stratified F1      : 0.1635
Decision Tree (depth=5) F1: 0.5556
Random Forest (100 trees) F1: 0.6072

→ These baselines tell you the floor.
→ Your current 0.77 is above these — now we need to find the ceiling.


In [7]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb   = np.zeros((len(X_train), 6))
test_lgb  = np.zeros((len(X_test),  6))

lgb_params = dict(
    objective='multiclass', num_class=6, metric='multi_logloss',
    n_estimators=2000, learning_rate=0.03, num_leaves=127,
    max_depth=-1, min_child_samples=10, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1,
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    Xtr, Xval = X_train[tr_idx], X_train[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
    
    oof_lgb[val_idx]  = model.predict_proba(Xval)
    test_lgb         += model.predict_proba(X_test) / N_SPLITS
    
    fold_f1 = f1_score(yval, np.argmax(oof_lgb[val_idx], axis=1), average='macro')
    print(f'Fold {fold+1} LGB F1: {fold_f1:.4f}')

lgb_oof_f1 = f1_score(y_train, np.argmax(oof_lgb, axis=1), average='macro')
print(f'\nLGB OOF F1: {lgb_oof_f1:.4f}')

Fold 1 LGB F1: 0.7327
Fold 2 LGB F1: 0.7136
Fold 3 LGB F1: 0.7459
Fold 4 LGB F1: 0.7418
Fold 5 LGB F1: 0.7246

LGB OOF F1: 0.7320


In [8]:
oof_xgb  = np.zeros((len(X_train), 6))
test_xgb = np.zeros((len(X_test),  6))

xgb_params = dict(
    objective='multi:softprob', num_class=6, eval_metric='mlogloss',
    n_estimators=2000, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    reg_alpha=0.1, reg_lambda=1.0,
    tree_method='hist', random_state=42, n_jobs=-1,
    early_stopping_rounds=100,   # <-- moved here
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    Xtr, Xval = X_train[tr_idx], X_train[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              verbose=False)          # <-- removed early_stopping_rounds from here
    
    oof_xgb[val_idx]  = model.predict_proba(Xval)
    test_xgb         += model.predict_proba(X_test) / N_SPLITS
    
    fold_f1 = f1_score(yval, np.argmax(oof_xgb[val_idx], axis=1), average='macro')
    print(f'Fold {fold+1} XGB F1: {fold_f1:.4f}')

xgb_oof_f1 = f1_score(y_train, np.argmax(oof_xgb, axis=1), average='macro')
print(f'\nXGB OOF F1: {xgb_oof_f1:.4f}')

Fold 1 XGB F1: 0.7343
Fold 2 XGB F1: 0.7320
Fold 3 XGB F1: 0.7584
Fold 4 XGB F1: 0.7571
Fold 5 XGB F1: 0.7453

XGB OOF F1: 0.7459


In [9]:
oof_cat  = np.zeros((len(X_train), 6))
test_cat = np.zeros((len(X_test),  6))

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def load_sequences(folder):
    seqs, labels, fids = [], [], []
    fpaths = sorted(folder.rglob('*.csv'))
    for fpath in tqdm(fpaths, desc=f'Loading {folder.name}'):
        df = pd.read_csv(fpath)
        arr = df[['mean_x','mean_y','mean_z','std_x','std_y','std_z']].values
        if len(arr) < 300:
            arr = np.pad(arr, ((0, 300-len(arr)), (0,0)), mode='edge')
        arr = arr[:300]
        seqs.append(arr)
        fid   = int(df['file_id'].iloc[0])
        label = int(df['label'].iloc[0]) if 'label' in df.columns else -1
        labels.append(label)
        fids.append(fid)
    return np.array(seqs, dtype=np.float32), np.array(labels), np.array(fids)

print('Loading train sequences...')
train_seq, train_lbl, _ = load_sequences(TRAIN_DIR)
print('Loading test sequences...')
test_seq,  test_lbl, _  = load_sequences(TEST_DIR)

# Normalize per-channel
mean_seq = train_seq.mean(axis=(0,1), keepdims=True)
std_seq  = train_seq.std(axis=(0,1),  keepdims=True) + 1e-9
train_seq = (train_seq - mean_seq) / std_seq
test_seq  = (test_seq  - mean_seq) / std_seq

# (N, 300, 6) → (N, 6, 300) for Conv1d
train_seq = train_seq.transpose(0, 2, 1)
test_seq  = test_seq.transpose(0,  2, 1)

# Add magnitude of mean-xyz and magnitude of std-xyz as 2 extra channels → (N, 8, 300)
def add_magnitude_channels(seqs):
    mean_mag = np.sqrt(seqs[:,0:1,:]**2 + seqs[:,1:2,:]**2 + seqs[:,2:3,:]**2)
    std_mag  = np.sqrt(seqs[:,3:4,:]**2 + seqs[:,4:5,:]**2 + seqs[:,5:6,:]**2)
    return np.concatenate([seqs, mean_mag, std_mag], axis=1)

train_seq = add_magnitude_channels(train_seq)
test_seq  = add_magnitude_channels(test_seq)
print(f'Sequence shapes — train: {train_seq.shape}, test: {test_seq.shape}')  # expect (N, 8, 300)

Device: cuda
Loading train sequences...


Loading train: 100%|██████████| 11020/11020 [00:30<00:00, 359.77it/s]


Loading test sequences...


Loading test: 100%|██████████| 6849/6849 [00:19<00:00, 343.61it/s]


Sequence shapes — train: (11020, 8, 300), test: (6849, 8, 300)


In [14]:
# --- Model definition ---

class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad)
        self.bn1   = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad)
        self.bn2   = nn.BatchNorm1d(channels)
        self.relu  = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


class MultiScaleCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=6, dropout=0.2):
        super().__init__()
        # Three parallel stems capture short, medium and long temporal patterns
        self.stem_small  = nn.Sequential(nn.Conv1d(n_channels, 64, kernel_size=3,  padding=1),  nn.BatchNorm1d(64), nn.ReLU())
        self.stem_medium = nn.Sequential(nn.Conv1d(n_channels, 64, kernel_size=7,  padding=3),  nn.BatchNorm1d(64), nn.ReLU())
        self.stem_large  = nn.Sequential(nn.Conv1d(n_channels, 64, kernel_size=15, padding=7),  nn.BatchNorm1d(64), nn.ReLU())

        # Merge 192 channels → 128, then downsample
        self.merge  = nn.Sequential(nn.Conv1d(192, 128, kernel_size=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2))   # 300→150
        self.stage1 = nn.Sequential(ResBlock(128, 3), ResBlock(128, 3), nn.MaxPool1d(2))                                   # 150→75
        self.stage2 = nn.Sequential(nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
                                    ResBlock(256, 3), ResBlock(256, 3), nn.MaxPool1d(3))                                   # 75→25
        self.stage3 = nn.Sequential(nn.Conv1d(256, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(),
                                    ResBlock(512, 3))

        # Global avg pool + global max pool concatenated → 1024 features
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.gmp = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        s = torch.cat([self.stem_small(x), self.stem_medium(x), self.stem_large(x)], dim=1)
        s = self.merge(s)
        s = self.stage1(s)
        s = self.stage2(s)
        s = self.stage3(s)
        out = torch.cat([self.gap(s), self.gmp(s)], dim=1)
        return self.classifier(out)


def augment_batch(x, noise_std=0.02, scale_range=(0.9, 1.1)):
    """Random noise + random amplitude scaling applied during training only."""
    noise = torch.randn_like(x) * noise_std
    scale = torch.empty(x.size(0), 1, 1, device=x.device).uniform_(*scale_range)
    return x * scale + noise


# --- Training loop ---

oof_cnn  = np.zeros((len(train_seq), 6))
test_cnn = np.zeros((len(test_seq),  6))

EPOCHS = 200
BATCH  = 64

Xt = torch.tensor(test_seq).to(DEVICE)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_seq, train_lbl)):
    print(f'\n--- Fold {fold+1}/{N_SPLITS} ---')

    Xtr     = torch.tensor(train_seq[tr_idx]).to(DEVICE)
    ytr     = torch.tensor(train_lbl[tr_idx]).long().to(DEVICE)
    Xval    = torch.tensor(train_seq[val_idx]).to(DEVICE)
    yval_np = train_lbl[val_idx]

    ds = TensorDataset(Xtr, ytr)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)

    model = MultiScaleCNN(n_channels=8, n_classes=6, dropout=0.2).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=EPOCHS, steps_per_epoch=len(dl))
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_f1, best_probs, best_state = 0, None, None

    for ep in range(EPOCHS):
        model.train()
        for xb, yb in dl:
            xb = augment_batch(xb)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.softmax(model(Xval), dim=1).cpu().numpy()
        ep_f1 = f1_score(yval_np, val_probs.argmax(1), average='macro')

        if ep_f1 > best_f1:
            best_f1    = ep_f1
            best_probs = val_probs
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep + 1) % 20 == 0:
            print(f'  Epoch {ep+1}/{EPOCHS} | F1: {ep_f1:.4f} | Best: {best_f1:.4f}')

    oof_cnn[val_idx] = best_probs
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_cnn += torch.softmax(model(Xt), dim=1).cpu().numpy() / N_SPLITS

    print(f'Fold {fold+1} CNN F1: {best_f1:.4f}')

cnn_oof_f1 = f1_score(train_lbl, oof_cnn.argmax(1), average='macro')
print(f'\nCNN OOF F1: {cnn_oof_f1:.4f}')


--- Fold 1/5 ---
  Epoch 20/200 | F1: 0.6192 | Best: 0.7365
  Epoch 40/200 | F1: 0.6531 | Best: 0.7365
  Epoch 60/200 | F1: 0.6677 | Best: 0.7365
  Epoch 80/200 | F1: 0.7275 | Best: 0.7433
  Epoch 100/200 | F1: 0.7139 | Best: 0.7433
  Epoch 120/200 | F1: 0.7187 | Best: 0.7433
  Epoch 140/200 | F1: 0.7213 | Best: 0.7433
  Epoch 160/200 | F1: 0.7271 | Best: 0.7433
  Epoch 180/200 | F1: 0.7309 | Best: 0.7433
  Epoch 200/200 | F1: 0.7328 | Best: 0.7433
Fold 1 CNN F1: 0.7433

--- Fold 2/5 ---
  Epoch 20/200 | F1: 0.6890 | Best: 0.7242
  Epoch 40/200 | F1: 0.6701 | Best: 0.7298
  Epoch 60/200 | F1: 0.6920 | Best: 0.7298
  Epoch 80/200 | F1: 0.6910 | Best: 0.7298
  Epoch 100/200 | F1: 0.7040 | Best: 0.7298
  Epoch 120/200 | F1: 0.7068 | Best: 0.7298
  Epoch 140/200 | F1: 0.7160 | Best: 0.7298
  Epoch 160/200 | F1: 0.6980 | Best: 0.7298
  Epoch 180/200 | F1: 0.7032 | Best: 0.7298
  Epoch 200/200 | F1: 0.7046 | Best: 0.7298
Fold 2 CNN F1: 0.7298

--- Fold 3/5 ---
  Epoch 20/200 | F1: 0.6829 | 

In [15]:
from scipy.optimize import minimize

def ensemble_f1(weights):
    w = np.array(weights)
    w = np.abs(w) / np.abs(w).sum()
    blended = (w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat + w[3]*oof_cnn)
    preds   = blended.argmax(axis=1)
    return -f1_score(y_train, preds, average='macro')

best_score = 1.0
best_weights = [0.25, 0.25, 0.25, 0.25]
for _ in range(30):
    w0 = np.random.dirichlet(np.ones(4))
    res = minimize(ensemble_f1, w0, method='Nelder-Mead',
                   options={'maxiter': 2000, 'xatol': 1e-5})
    if res.fun < best_score:
        best_score   = res.fun
        best_weights = res.x

w = np.abs(best_weights) / np.abs(best_weights).sum()
print(f'Optimal weights  LGB={w[0]:.3f}  XGB={w[1]:.3f}  CAT={w[2]:.3f}  CNN={w[3]:.3f}')
oof_ensemble = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat + w[3]*oof_cnn
print(f'Ensemble OOF F1: {f1_score(y_train, oof_ensemble.argmax(1), average="macro"):.4f}')

Optimal weights  LGB=0.004  XGB=0.408  CAT=0.364  CNN=0.224
Ensemble OOF F1: 0.7579


In [16]:
test_ensemble = w[0]*test_lgb + w[1]*test_xgb + w[2]*test_cat + w[3]*test_cnn
test_preds    = test_ensemble.argmax(axis=1)

sample_sub = pd.read_csv(SAMPLE)
# build a mapping from file_id to prediction
pred_map = dict(zip(test_ids.tolist(), test_preds.tolist()))
sample_sub['Label'] = sample_sub['Id'].map(pred_map)

# safety check
assert sample_sub['Label'].isna().sum() == 0, "Some IDs were not predicted!"

sample_sub.to_csv('submission.csv', index=False)
print('submission.csv written')
print(sample_sub['Label'].value_counts().sort_index())

submission.csv written
Label
0    2834
1    3202
2      27
3     514
4      57
5     215
Name: count, dtype: int64
